# Secure Device Identity

In this exercise, unique identifiers of an embedded system are explored and an authentication scheme based on symmetric cryptography is analyzed.

## Preparation

For shell command calls, the Python module ``subprocess`` is used (https://docs.python.org/3/library/subprocess.html).

Run the following code block to provide the functions ``execute_vm_command()`` and ``execute_stm32_command()`` within this notebook.

In [ ]:
import subprocess

def execute_vm_command(cmd):
    run = subprocess.run(cmd, universal_newlines=True, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(run.stdout)
    
def execute_stm32_command(cmd):
    ssh = 'sshpass -p "root" ssh root@192.168.13.13 '
    run = subprocess.run((ssh + cmd), universal_newlines=True, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(run.stdout)

Of course, all commands given in this tutorial can also be executed directly on within a Linux console.

If you want to open a terminal window, execute the next code block.

In [ ]:
execute_vm_command('gnome-terminal')

---

## Exercise 6.1: Identify the Identifiers

An important first step to establish a device's identity is the identification of suitable, unique identifiers of said system.

### MAC Address(es)

The Media Access Control (MAC) address is a common unique identifier assigned to a Network Interface Controller (NIC). It is sometimes used by manufacturers to identify the whole device.

On Linux-based systems, the following command shows MAC address for the network interface ``eth0``.

In [ ]:
execute_stm32_command('cat /sys/class/net/eth0/address')

The Wi-Fi module ``wlan0`` also has a unique MAC address.

In [ ]:
execute_stm32_command('cat /sys/class/net/wlan0/address')

So does the Bluetooth module.

In [ ]:
execute_stm32_command('cat /sys/kernel/debug/bluetooth/hci0/identity')

### SD Card ID

Hard disks, flash memories, and memory cards usually also carry unique numbers for identification.

For SD cards, the Card Identification (CID) register is a 16 byte code uniquely identifying a specific card.

On the given Linux system, it can be read as shown in the following command.

In [ ]:
execute_stm32_command('cat /sys/block/mmcblk0/device/cid')

Think about which disadvantages unique IDs of hard disks and memory cards could bring with them.

**Disadvantages of Unique IDs of Hard Disks and Memory Cards:**

*TODO*

### Unique Identifier of CPUs and Microcontrollers

Usually, microchip like CPUs and microcontrollers also carry a unique identification number.

For the STM32MP1, the reference manual states:<br>
''The 96-bit unique device identifier provides a reference number, unique for a given device and in any context. These bits cannot be altered by the user.''

This UID is immutably stored in the One-Time Programmable (OTP) memory of the STM32MP1 chip. It is split into three 32-bit words that can be found at the following addresses:

```
Base address: 0x5C00 5000 (BSEC base address on APB5)
Address offset: 0x234 = UID[31:0]
Address offset: 0x238 = UID[63:32]
Address offset: 0x23C = UID[95:64]
```

Use the tool ``devmem`` directly on your STM32MP1 board to read the content of the OTP and reconstruct the UID.

**STM32MP1 Unique Device ID:**

*TODO*

Also, the ``/proc/cpuinfo`` might be of help for CPU identifiers.

Compare the results to the previous values read by ``devmem``.

In [ ]:
execute_stm32_command('cat /proc/cpuinfo')

### Device/System Identity

Discuss which identifiers you would use to identify your embedded device and if it makes sense to use multiple values!

**Best Identifiers and Reasons:**
    
*TODO*

**Reasons for/against Usage of Multiple Identifiers:**

*TODO*

---

## Exercise 6.2: Identity Generation and Distribution

Imagine, your team decided to use the serial number provided by ``/proc/cpuinfo`` as the unique identifier for your embedded device and plans to authenticate your device by the Challenge-Handshake Authentication Protocol (CHAP).

The following steps represent an example implementation of the generation and distribution phase.

First, let's create the directory ``/home/root/secure_device_identity`` on your STM32MP1 device.

In [ ]:
execute_stm32_command('mkdir -p /home/root/secure_device_identity')

### Generation Phase

First, the unique identifier has to be collected and saved.

The Python script ``scripts/get_identity.py`` contains the following code to achieve that.

```python
#!/usr/bin/env python3

import subprocess

output = subprocess.Popen('cat /proc/cpuinfo | grep Serial', shell=True, stdout=subprocess.PIPE)

unique_id      = output.stdout.read().split()[2]
unique_id_file = 'identity'

with open(unique_id_file, 'wb') as f:
  f.write(unique_id)

print('Collected unique ID:        ' + unique_id.decode())
print('Stored unique ID in file:   ' + unique_id_file)
```

Use the following code block to copy the script to your board and make it executable.

In [ ]:
execute_vm_command('sshpass -p "root" scp scripts/get_identity.py root@192.168.13.13:/home/root/secure_device_identity')
execute_stm32_command('chmod +x /home/root/secure_device_identity/get_identity.py')

Then, change to board, navigate to the script's directory and execute it:

```bash
cd /home/root/secure_device_identity/
./get_identity.py

```

The second step is to generate a suitable authenticator for the desired cryptographic authentication scheme.

In this case, the Python script ``scripts/create_key.py`` implements the following code to generate a 32-byte symmetric secret for the CHAP scheme.

```python
#!/usr/bin/env python3

import secrets

auth_key      = secrets.token_hex(32)
auth_key_file = 'authkey'

with open(auth_key_file, 'w') as f:
  f.write(auth_key)

print('Created 32-byte secret key: ' + auth_key)
print('Stored key in file:         ' + auth_key_file)
```

Use the following code block to also copy the second script to your board and make it executable.

In [ ]:
execute_vm_command('sshpass -p "root" scp scripts/create_key.py root@192.168.13.13:/home/root/secure_device_identity')
execute_stm32_command('chmod +x /home/root/secure_device_identity/create_key.py')

Then, again, change to board, navigate to the script's directory and execute it:

```bash
cd /home/root/secure_device_identity/
./create_key.py

```

### Distribution Phase

In this case, the unique ID data and the authentication secret are generated on the device and have to be read-out from it and stored in the corresponding identity management system (this machine).

Use the following commands to copy the board identity data and authentication key to this machine. This is necessary to allow for device authentication at a later stage.

In [ ]:
execute_vm_command('sshpass -p "root" scp root@192.168.13.13:/home/root/secure_device_identity/identity ./scripts/')
execute_vm_command('sshpass -p "root" scp root@192.168.13.13:/home/root/secure_device_identity/authkey ./scripts/')

Finally, check if the copied data is valid.

In [ ]:
execute_vm_command('echo "Identity:" && cat scripts/identity')
execute_vm_command('echo "\nAuthentication Key:" && cat scripts/authkey')

### Discussion

Discuss, if an authentication scheme based on asymmetric crypto would have advantages and which changes would be necessary in the generation and the distribution phase pf the previous example.

**Advantages of Authentication based on Asymmetric Crypto**
    
*TODO*

**Necessary Changes to the Generation Phase**
    
*TODO*

**Necessary Changes to the Distribution Phase**
    
*TODO*

---

## Exercise 6.3: Authentication Protocol

After generating and distributing identity and authentication data, the device is ready to be ''sold''.

In the field, the device can request authentication as shown in the following example implementation based on CHAP. Here, as a trivial scenario, the (client) device requests to receive the current system time of the server, which is only possible after authentication (in this example).

### Authentication Process

First, take a look a the source code of client and server to reconstruct the steps of the CHAP scheme.

**Source Code `scripts/client.py`:**

```python
#!/usr/bin/python3

import socket
import hashlib

ip = '192.168.13.17'

with open('identity', 'rb') as idfile:
    id = idfile.read()

with open('authkey', 'rb') as authkey:
    key = authkey.read()

clientsocket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
clientsocket.connect((ip, 5555))
print('Client started ...')

print('Sending authentication request to authentication server in order to receive current system time ...')
clientsocket.send(id)

challenge = clientsocket.recv(64)
print('Received challenge: ' + challenge.decode())

response = hashlib.sha256(challenge + key).hexdigest()
print('Created response: ' + response)

print('Sending response back to authentication server ...')
clientsocket.send(response.encode())

print('Waiting for authentication ...')
authmsg = clientsocket.recv(50)
if authmsg.decode() == 'AUTHENTICATION ERROR!':
    print('AUTHENTICATION ERROR!')
else:
    print('AUTHENTICATION SUCCESS!')
    print('Received authentication server system time: ' + authmsg.decode())

clientsocket.close()
```

**Source Code ``scripts/server.py``**

```python
#!/usr/bin/python3

import socket
import secrets
import hashlib
from datetime import datetime, timezone

ip = '192.168.13.17'

with open('identity', 'rb') as idfile:
    id = idfile.read()

serversocket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
serversocket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
serversocket.bind((ip, 5555))
serversocket.listen(5)
print('Server started ...')

while True:
    clienthandle, addr = serversocket.accept()
    print('Connection from %s' % str(addr))
    
    recv_uid = clienthandle.recv(24)
    print('Received UID: ' + recv_uid.decode())
    
    if recv_uid == id:
        print('Received UID matches known device ID!')

        challenge = secrets.token_hex(32)
        print('Created challenge: ' + challenge)
      
        print('Sending challenge to device ...')
        clienthandle.send(challenge.encode())
      
        with open('authkey', 'rb') as authkey:
            key = authkey.read()
        
        response = clienthandle.recv(64).decode()
        print('Received response: ' + response)
        
        response_expected = hashlib.sha256(challenge.encode() + key).hexdigest()
        print('Expected response: ' + response_expected)
        
        if response == response_expected:
            print('AUTHENTICATION SUCCESS!')
            now = datetime.now(timezone.utc).astimezone()
            timestamp = now.strftime('%Y/%m/%d %H:%M:%S')
            print('Current time: ' + timestamp)
            clienthandle.send(timestamp.encode())
            clienthandle.close()
            break

        else:
            print('AUTHENTICATION ERROR: response does not match expected value!')
            clienthandle.send(b'AUTHENTICATION ERROR!')
            clienthandle.close()

    else:
        print('AUTHENTICATION ERROR: received UID does not match known device ID!')
        clienthandle.send(b'AUTHENTICATION ERROR!')
        clienthandle.close()
```

**Steps of Authentication Protocol:**
    
*TODO*

### Running the Example Implementation

Open a terminal with the following command.

In [ ]:
execute_vm_command('gnome-terminal')

Then, execute the following commands in your terminal to start the authentication server:

```bash
cd scripts
./server.py
```

Afterward, copy the client script to your device:

In [ ]:
execute_vm_command('sshpass -p "root" scp scripts/client.py root@192.168.13.13:/home/root/secure_device_identity')
execute_stm32_command('chmod +x /home/root/secure_device_identity/client.py')

Change to the terminal on your board, navigate to the script's directory, and initiated the authentication procedure:

```bash
cd /home/root/secure_device_identity/
./client.py

```

### Discussion

How would you test the correct functioning of the authentication procedure?

**Important Tests for Authentication Implementation:**

*TODO*